# HD 137010 single transit — prior sensitivity

`prior_sensitivity.yaml` is one run of 12 chains: the 5 eccentricity priors with the
single-transit period prior, plus `circular` with a uniform period prior as the
reference, 2 seeds each.

In [ ]:
import bay
import numpy as np
import matplotlib.pyplot as plt


In [ ]:

a = 267
p = bay.TransitModel(t0=0.0, per=350.0, rp=0.0136, inc=bay.inc_from_b(0.28, a), a=a,
                     ecc=0.0, w=90.0, u1=0.56, u2=0.10)

print(f"b             = {p.b:.3f}")
print(f"R_p           = {p.R_planet_earth:.2f} R_earth")
print(f"P             = {p.period_years:.3f} yr")
print(f"a             = {p.a_au:.3f} AU")
print(f"rho_implied   = {p.rho_star_implied:.2f} g/cm^3")
print(f"P from rho    = {p.per_from_density:.1f} d")
print(f"insolation    = {p.insolation:.2f} S_earth")
print(f"HZ cons       = {p.hz_conservative_au} AU")

# Same P, a, inc but eccentric, transit at periastron
q = bay.TransitModel(t0=0.0, per=350.0, rp=0.0136, inc=p.inc, a=a,
                     ecc=0.3, w=90.0, u1=0.56, u2=0.10)
print(f"\ne=0.3: b = {q.b:.3f}, r = {q.r_min_au:.3f} .. {q.r_max_au:.3f} AU, "
      f"insolation = {q.insolation:.2f} S_earth")

## Run the recipe

One process per chain. The run directory gets a copy of the recipe and one
`chain_<name>.npz` per chain.

In [ ]:
# chain_paths = bay.run("prior_sensitivity.yaml")
# chain_paths = bay.run("mcmc_tuning.yaml")
# chain_paths = bay.run("seed_consistency.yaml")


from pathlib import Path

RUN = sorted(Path("results").glob("*_prior-sensitivity"))[-1]   # latest run
chain_paths = sorted(str(p) for p in RUN.glob("chain_*.npz"))
print(RUN, *[Path(p).name for p in chain_paths], sep="\n  ")

## Analysis

Per chain: drop the first 20% of steps, then

    R_star        from (log10_stellar_mass, log10_stellar_density)
    a_AU          from (orbital_period, log10_stellar_density) x R_star
    insolation    S = L / (a_AU^2 sqrt(1 - e^2)),   L ~ N(0.232, 0.022) L_sun

Habitable-zone edges in S (Kopparapu et al. 2013, as quoted by Venner et al. 2026):
conservative [0.30, 1.00], optimistic [0.27, 1.60].

In [ ]:
import numpy as np
import pandas as pd

from bay import AU, IDX, M_SUN, R_SUN, a_from_rho, load

STELLAR_LUMINOSITY_MU, STELLAR_LUMINOSITY_SIGMA = 0.232, 0.022   # L_sun
HZ_CONSERVATIVE = (0.30, 1.00)      # S_earth
HZ_OPTIMISTIC = (0.27, 1.60)
BURN_FRACTION = 0.2


def flat_chain(path, burn_fraction=BURN_FRACTION):
    """Load an .npz chain and flatten it, dropping the first 20% of steps."""
    chain = load(path)["chain"]                 # (nsteps_thinned, nwalkers, ndim)
    start = int(burn_fraction * chain.shape[0])
    return chain[start:].reshape(-1, chain.shape[-1]).astype(float)


def derived(flat, circular=False, rng=None):
    """Physical quantities from a flat chain.

    In a circular run h and q are sampled but never reach the light curve, so
    they come back as their prior; the orbit is a circle and the insolation has
    to be evaluated at e = 0, not at the prior draw.
    """
    rng = np.random.default_rng(rng)

    orbital_period = np.exp(flat[:, IDX["ln_orbital_period"]])
    stellar_density = 10.0 ** flat[:, IDX["log10_stellar_density"]]
    stellar_mass = 10.0 ** flat[:, IDX["log10_stellar_mass"]]
    eccentricity = (flat[:, IDX["sqrt_eccentricity_cos_periastron_argument"]] ** 2
                    + flat[:, IDX["sqrt_eccentricity_sin_periastron_argument"]] ** 2)
    if circular:
        eccentricity = np.zeros_like(eccentricity)

    # R_star [R_sun] from M = 4/3 pi R^3 rho
    stellar_radius = (3 * stellar_mass * M_SUN
                      / (4 * np.pi * stellar_density)) ** (1 / 3) / R_SUN

    semi_major_axis_in_stellar_radii = a_from_rho(orbital_period, stellar_density)
    semi_major_axis_au = semi_major_axis_in_stellar_radii * stellar_radius * R_SUN / AU

    stellar_luminosity = rng.normal(STELLAR_LUMINOSITY_MU, STELLAR_LUMINOSITY_SIGMA,
                                    size=flat.shape[0])
    orbit_averaged_insolation = (stellar_luminosity
                                 / (semi_major_axis_au ** 2 * np.sqrt(1 - eccentricity ** 2)))

    return {
        "orbital_period": orbital_period,
        "eccentricity": eccentricity,
        "stellar_radius": stellar_radius,
        "semi_major_axis_au": semi_major_axis_au,
        "orbit_averaged_insolation": orbit_averaged_insolation,
    }


def summarise(path, rng=None):
    c = load(path)
    circular = c["spec"]["priors"]["eccentricity"]["dist"] == "circular"
    d = derived(flat_chain(path), circular=circular, rng=rng)
    insolation = d["orbit_averaged_insolation"]
    cons_lo, cons_hi = HZ_CONSERVATIVE
    opt_lo, opt_hi = HZ_OPTIMISTIC
    return {
        "chain": c["spec"]["name"],
        "acc": float(np.mean(c["acc"])),
        "median_P_d": np.median(d["orbital_period"]),
        "median_e": np.median(d["eccentricity"]),
        "median_a_au": np.median(d["semi_major_axis_au"]),
        "median_S": np.median(insolation),
        "P_HZ_cons": np.mean((insolation >= cons_lo) & (insolation <= cons_hi)),
        "P_HZ_opt": np.mean((insolation >= opt_lo) & (insolation <= opt_hi)),
        "hot_cons": np.mean(insolation > cons_hi),
        "cold_cons": np.mean(insolation < cons_lo),
        "hot_opt": np.mean(insolation > opt_hi),
        "cold_opt": np.mean(insolation < opt_lo),
    }


table = pd.DataFrame([summarise(p, rng=0) for p in chain_paths])
table.round(4)

## Convergence and stuck walkers (checks 4 and 5)

In [ ]:
import emcee
from pathlib import Path

for path in chain_paths:
    c = load(path)
    chain, last, thin = c["chain"].astype(float), c["log_prob"][-1], c["spec"]["thin"]
    spread = np.median(last) - last.min()
    tau = emcee.autocorr.integrated_time(chain, quiet=True) * thin
    nsteps = chain.shape[0] * thin
    print(f"{Path(path).name:34s} max tau {tau.max():8.1f}  "
          f"(limit {nsteps / 50:.0f})   worst-walker gap {spread:7.2f}")

## Best fit and prior vs posterior

In [ ]:
import matplotlib.pyplot as plt

reference = load(chain_paths[0])
spec = reference["spec"]
DATA = "../data/apjladf06ft3_mrt.txt"  
fit = bay.Posterior(spec, DATA)

chain = reference["chain"].astype(float)
flat, flat_log_prob = chain.reshape(-1, chain.shape[-1]), reference["log_prob"].reshape(-1)
best = flat[np.argmax(flat_log_prob)]

model = bay.TransitModel.from_theta(best, fit.circular)
out_of_transit_flux = best[IDX["out_of_transit_flux"]]
white_noise_amplitude = np.exp(best[IDX["ln_white_noise_amplitude"]])
t_fine = np.linspace(fit.t.min(), fit.t.max(), 2000)

plt.errorbar(fit.t, fit.flux, yerr=white_noise_amplitude, fmt=".k", alpha=0.5)
plt.plot(t_fine, out_of_transit_flux * model.light_curve(t_fine), "r")
plt.xlabel("time [BKJD]")
plt.ylabel("flux")
plt.title(spec["name"])
plt.show()

In [ ]:
from bay import NAMES

posterior_flat = flat_chain(chain_paths[0])
prior_flat = bay.sample_prior(spec["priors"], 20000, rng=1)

def with_derived(sample):
    d = {name: sample[:, IDX[name]] for name in NAMES}
    d["eccentricity"] = (d["sqrt_eccentricity_cos_periastron_argument"] ** 2
                         + d["sqrt_eccentricity_sin_periastron_argument"] ** 2)
    d["orbital_period [d]"] = np.exp(d["ln_orbital_period"])
    return d

posterior, prior = with_derived(posterior_flat), with_derived(prior_flat)

fig, axes = plt.subplots(4, 4, figsize=(16, 13))
for ax, name in zip(axes.flat, posterior):
    ax.hist(prior[name], bins=60, density=True, color="lightgrey", label="prior")
    ax.hist(posterior[name], bins=60, density=True, histtype="step", color="k",
            label="posterior")
    ax.set_title(name, fontsize=8)
for ax in axes.flat[len(posterior):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=7)
plt.tight_layout()
plt.show()


"""
Plots to add to package
- corner plots with prior posterior distributions in diagonal
- posterior combinations period vs eccentricity
- parameters direct distribution comparison between eccentricity priors
- model consistency and autocorrelation time
- random seed vs convergence consistency
"""

### Bayes factor, circular vs eccentric

Savage-Dickey at e = 0: the posterior density near e = 0 over the prior density
near e = 0, from the same chain. It tests e = 0 as a point null *inside* each
eccentric model, so the `circular` runs play no part in it.

Two sanity conditions on the number. It should not depend on the window `eps`
(if it does, the window is doing the work, not the data -- and for
`beta_kipping13` the prior density diverges as e^-0.133 at e -> 0, so this is
worth checking). And the two seeds bracket the Monte Carlo error.

In [ ]:
EPS_WINDOWS = (0.01, 0.02, 0.05, 0.10)
PRIOR_DRAWS = 100_000

import json


def eccentricity(sample):
    return (sample[:, IDX["sqrt_eccentricity_cos_periastron_argument"]] ** 2
            + sample[:, IDX["sqrt_eccentricity_sin_periastron_argument"]] ** 2)


prior_cache = {}


def prior_eccentricity(priors):
    """Prior draws of e, cached -- both seeds of a run share the same prior."""
    key = json.dumps(priors, sort_keys=True)
    if key not in prior_cache:
        prior_cache[key] = eccentricity(bay.sample_prior(priors, PRIOR_DRAWS, rng=1))
    return prior_cache[key]


def savage_dickey(path):
    spec = load(path)["spec"]
    if spec["priors"]["eccentricity"]["dist"] == "circular":
        return None            # e is not a parameter of that model

    posterior_e = eccentricity(flat_chain(path))
    prior_e = prior_eccentricity(spec["priors"])

    row = {"chain": spec["name"]}
    for eps in EPS_WINDOWS:
        row[f"BF(eps={eps:g})"] = (np.mean(posterior_e < eps)
                                   / np.mean(prior_e < eps))
    row["n_in_window"] = int(np.sum(posterior_e < EPS_WINDOWS[1]))
    row["median_e_post"] = np.median(posterior_e)
    row["median_e_prior"] = np.median(prior_e)
    return row


bayes_factors = pd.DataFrame([row for row in map(savage_dickey, chain_paths)
                              if row is not None])
bayes_factors.round(3)

In [ ]:
# --- cross plot ----------------------------------------------------------
import corner

PARAMS = ["orbital_period [d]", "mid_transit_time", "impact_parameter",
          "ln_planet_to_star_radius_ratio", "log10_stellar_density",
          "log10_stellar_mass", "eccentricity"]

if "posterior" not in globals():
    posterior = with_derived(flat_chain(chain_paths[0]))

samples = np.column_stack([posterior[p] for p in PARAMS])
fig = corner.corner(
    samples, labels=PARAMS)
